# COMP5339 Assignment 2 (Group 156) — Task 5: Data Subscription & Map Dashboard
**Group 156** — Member 1 (540989498) · Member 2 (540929966) · Member 3 (540903678)


This is the **subscriber** half of the system. It subscribes to the MQTT topic the
publisher notebook (`Assignment2_Group156.ipynb`, Tasks 1–3/6) streams to, and renders a
**live map-based dashboard** with Plotly Dash.

**Features:**
- Live-updating map: a marker per facility at its real location, refreshed as
  messages arrive.
- Metric **toggle** between power output and CO2 emissions (marker colour + size).
- **Filters** by network region and fuel technology.
- Hover + **click pop-up** showing station name, fuel type, latest power & emissions.
- Network **price & demand** read from the optional Task 1 market data.
- **Integration with Assignment 1**: every received message is matched to its
  `FACILITY` row (by `facility_code`) and written into the shared DuckDB schema
  (`FACILITY_POWER_EMISSIONS`).

**Install dependencies:** run `pip install -r requirements.txt` in the folder that contains `requirements.txt`. (You can just remove the comment of the first code cell and run it)

**How to run:**
1. In `Assignment2_Group156.ipynb`, run Tasks 1–4 (so the CSVs + `electricity_a2.duckdb` exist).
2. Run all cells **here** — the Dash app opens at http://127.0.0.1:8050 and waits for data.
3. Back in `Assignment2_Group156.ipynb`, run Task 6 (`run_continuous_stream`; set
   `rounds=None` for an unbounded live stream). Watch markers populate the map live.

> Run the two notebooks in **separate kernels**. They communicate only through the
> MQTT broker; the dashboard also writes received records back into the shared
> `electricity_a2.duckdb`.


In [1]:
#%pip install -r requirements.txt

In [2]:
import os
import json
import time
import threading
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import paho.mqtt.client as mqtt
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, no_update

try:
    import duckdb
    HAS_DUCKDB = True
except ImportError:
    HAS_DUCKDB = False


In [3]:

MQTT_HOST = "broker.hivemq.com"
MQTT_PORT = 1883
MQTT_TOPIC = "comp5339/group156/electricity"
MQTT_SUB_CLIENT_ID = f"comp5339-group156-dash-{int(time.time())}"

DATA_DIR = Path("data")
DB_PATH = DATA_DIR / "electricity_a2.duckdb"
MARKET_CSV = DATA_DIR / "market_price_demand_5min.csv"
FACILITIES_JSON = DATA_DIR / "raw_openelectricity" / "facilities.json"

DASH_PORT = 8050
INTEGRATE_TO_DUCKDB = True   # write received records into the A1/A2 schema

# Marker colour scales per metric.
COLOR_SCALES = {"power_mw": "Viridis", "emissions_tco2e": "Reds"}

In [4]:

facility_meta = pd.DataFrame()
code_to_facility_id = {}

if HAS_DUCKDB and DB_PATH.exists():
    try:
        _c = duckdb.connect(str(DB_PATH), read_only=True)
        facility_meta = _c.execute("""
            SELECT f.facility_code, f.facility_name, f.state, f.network_region,
                   f.lat, f.lon, f.capacity_registered_mw,
                   ft.fuel_name AS fueltech_id
            FROM FACILITY f
            LEFT JOIN FUEL_TYPE ft ON ft.fuel_id = f.fuel_id
        """).df()
        idmap = _c.execute("SELECT facility_code, facility_id FROM FACILITY").df()
        code_to_facility_id = dict(zip(idmap["facility_code"], idmap["facility_id"]))
        _c.close()
        print(f"Loaded {len(facility_meta)} facilities from DuckDB schema")
    except Exception as e:
        print(f"Could not read DuckDB ({e}); falling back to facilities.json")

if facility_meta.empty and FACILITIES_JSON.exists():
    raw = json.loads(FACILITIES_JSON.read_text(encoding="utf-8"))
    rows = []
    for fac in raw.get("data", []):
        loc = fac.get("location") or {}
        units = fac.get("units") or [{}]
        ft = next((u.get("fueltech_id") for u in units if u.get("fueltech_id")), None)
        rows.append({
            "facility_code": fac.get("code"),
            "facility_name": fac.get("name"),
            "network_region": fac.get("network_region"),
            "lat": loc.get("lat"), "lon": loc.get("lng") or loc.get("lon"),
            "fueltech_id": ft,
        })
    facility_meta = pd.DataFrame(rows)
    print(f"Loaded {len(facility_meta)} facilities from facilities.json")

print(facility_meta)
REGION_OPTIONS = sorted(facility_meta["network_region"].dropna().unique().tolist())
FUEL_OPTIONS = sorted(facility_meta["fueltech_id"].dropna().unique().tolist())

# Optional market series, indexed by event time for quick lookup.
market_df = pd.DataFrame()
if MARKET_CSV.exists():
    market_df = pd.read_csv(MARKET_CSV, parse_dates=["interval_ts"])
    market_df["interval_ts"] = pd.to_datetime(market_df["interval_ts"], utc=True)
    market_df = market_df.set_index("interval_ts").sort_index()
    print(f"Loaded market series: {len(market_df)} rows, "
          f"cols={[c for c in market_df.columns if c != 'network_region']}")

Loaded 542 facilities from DuckDB schema
    facility_code          facility_name state network_region        lat  \
0         0WDBESS           Williamsdale   NSW           NSW1 -35.574580   
1           0WNSF           Winton North   VIC           VIC1 -36.499717   
2             ADP  Adelaide Desalination    SA            SA1 -35.100751   
3          AGLHAL                Hallett    SA            SA1 -33.349528   
4         AGLNOW1             West Nowra   NSW           NSW1 -34.882318   
..            ...                    ...   ...            ...        ...   
537        YATSF1                Yatpool   VIC           VIC1 -34.396363   
538         YAWWF                 Yawong   VIC           VIC1 -36.471022   
539      YENDONWF                 Yendon   VIC           VIC1 -37.630407   
540          YSWF           Yaloak South   VIC           VIC1 -37.716474   
541      YWNGAHYD             Yarrawonga   VIC           VIC1 -36.009466   

            lon  capacity_registered_mw       

In [5]:
# MQTT subscriber and Shared State

STATE_LOCK = threading.Lock()
latest_by_facility = {}        # facility_code -> latest message dict
stats = defaultdict(int)       # simple counters for the status bar
latest_event_time = {"ts": None}

# --- Rolling history buffer for time-series charts -------------------
# Keyed by interval_ts (ISO string), value is dict[facility_code]->record.
# Capped to the most recent HISTORY_MAX_INTERVALS distinct event times,
# so memory stays bounded even on an unbounded stream.
from collections import OrderedDict
HISTORY_MAX_INTERVALS = 24 * 12   # last 24h of 5-min intervals
history_by_interval = OrderedDict()


# Optional: a writer connection into the DuckDB schema (integration bonus).
_db_con = None
_obs_counter = {"next": 1}
if INTEGRATE_TO_DUCKDB and HAS_DUCKDB and DB_PATH.exists():
    try:
        _db_con = duckdb.connect(str(DB_PATH))   # read-write
        _mx = _db_con.execute(
            "SELECT COALESCE(MAX(obs_id), 0) FROM FACILITY_POWER_EMISSIONS"
        ).fetchone()[0]
        _obs_counter["next"] = int(_mx) + 1
        print(f"DuckDB integration ON — next obs_id={_obs_counter['next']}")
    except Exception as e:
        print(f"DuckDB integration disabled ({e})")
        _db_con = None


def _persist_to_duckdb(rec):
    """Write one received record into FACILITY_POWER_EMISSIONS, matched to the
    correct FACILITY row by facility_code (Assignment 1 integration).
    Skips records already present to avoid duplicates when replaying history."""
    if _db_con is None:
        return
    fid = code_to_facility_id.get(rec.get("facility_code"))
    if fid is None:
        return
    ts = rec.get("event_time")
    try:
        if _db_con.execute(
            "SELECT 1 FROM FACILITY_POWER_EMISSIONS "
            "WHERE facility_id=? AND interval_ts=? LIMIT 1",
            [int(fid), ts]
        ).fetchone():
            return
    except Exception:
        pass
    with STATE_LOCK:
        oid = _obs_counter["next"]
        _obs_counter["next"] += 1
    try:
        _db_con.execute(
            "INSERT INTO FACILITY_POWER_EMISSIONS VALUES (?, ?, ?, ?, ?)",
            [oid, int(fid), ts, rec.get("power_mw"), rec.get("emissions_tco2e")],
        )
    except Exception:
        pass   # never let a DB hiccup break the live stream


def _on_connect(client, userdata, flags, rc, properties=None):
    print(f"[subscriber] connected rc={rc}, subscribing to {MQTT_TOPIC}")
    client.subscribe(MQTT_TOPIC, qos=0)


def _on_message(client, userdata, msg):
    try:
        rec = json.loads(msg.payload.decode("utf-8"))
    except Exception:
        with STATE_LOCK:
            stats["decode_errors"] += 1
        return
    fc = rec.get("facility_code")
    if not fc:
        return
    ts = rec.get("event_time")
    with STATE_LOCK:
        latest_by_facility[fc] = rec
        stats["received"] += 1
        latest_event_time["ts"] = ts
        if ts is not None:
            bucket = history_by_interval.get(ts)
            if bucket is None:
                bucket = {}
                history_by_interval[ts] = bucket
                # Cap history to last HISTORY_MAX_INTERVALS intervals.
                while len(history_by_interval) > HISTORY_MAX_INTERVALS:
                    history_by_interval.popitem(last=False)
            bucket[fc] = rec
    _persist_to_duckdb(rec)


try:
    sub = mqtt.Client(client_id=MQTT_SUB_CLIENT_ID,
                      callback_api_version=mqtt.CallbackAPIVersion.VERSION2)
except (AttributeError, TypeError):
    sub = mqtt.Client(client_id=MQTT_SUB_CLIENT_ID)
sub.on_connect = _on_connect
sub.on_message = _on_message
sub.connect(MQTT_HOST, MQTT_PORT, keepalive=60)
sub.loop_start()   # background network thread
print("Subscriber started — waiting for the publisher to stream messages...")


DuckDB integration ON — next obs_id=668135
Subscriber started — waiting for the publisher to stream messages...


In [6]:


# Coordinate lookup built once from the facility metadata loaded
# in cell 3. We use this as a fallback in snapshot_df() so the map
# still places a marker even if a particular MQTT message happens
# to ship without lat/lon (defensive — the publisher does send
# them, but JSON encoding of NaN-prone columns is worth guarding).
code_to_latlon: dict = {}
if not facility_meta.empty:
    code_to_latlon = {
        r["facility_code"]: (float(r["lat"]), float(r["lon"]))
        for r in facility_meta.dropna(subset=["lat", "lon"]).to_dict("records")
    }


def snapshot_df():
    """Thread-safe snapshot of the latest record per facility as a dataframe.

    Falls back to ``code_to_latlon`` (built from the facility metadata) when
    a message arrives without coords, so the map is never silently empty.
    """
    with STATE_LOCK:
        records = list(latest_by_facility.values())
        n_recv = stats["received"]
        ev_ts = latest_event_time["ts"]
    if not records:
        return pd.DataFrame(), n_recv, ev_ts
    df = pd.DataFrame(records)
    for col in ("lat", "lon", "power_mw", "emissions_tco2e"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if code_to_latlon and "facility_code" in df.columns:
        missing_lat = df["lat"].isna() | df["lon"].isna()
        if missing_lat.any():
            df.loc[missing_lat, "lat"] = df.loc[missing_lat, "facility_code"].map(
                lambda c: code_to_latlon.get(c, (None, None))[0]
            )
            df.loc[missing_lat, "lon"] = df.loc[missing_lat, "facility_code"].map(
                lambda c: code_to_latlon.get(c, (None, None))[1]
            )
    df = df.dropna(subset=["lat", "lon"])
    return df, n_recv, ev_ts


def market_now(ev_ts):
    """Latest known price + demand at/just before the given event time."""
    if market_df.empty or ev_ts is None:
        return None, None
    try:
        ts = pd.Timestamp(ev_ts)
        sub_df = market_df.loc[:ts]
        if sub_df.empty:
            return None, None
        row = sub_df.iloc[-1]
        price = row.get("price") if "price" in market_df.columns else None
        demand = row.get("demand") if "demand" in market_df.columns else None
        return price, demand
    except Exception:
        return None, None


def history_df():
    """Flatten the rolling history buffer into a long-format dataframe.

    Returns a DataFrame with columns
    `interval_ts, facility_code, fueltech_id, network_region, power_mw,
    emissions_tco2e`. Empty if no messages have been received yet.
    """
    with STATE_LOCK:
        # Shallow-copy under the lock; values are small dicts.
        snapshot = [(ts, list(bucket.values()))
                    for ts, bucket in history_by_interval.items()]
    if not snapshot:
        return pd.DataFrame()
    rows = []
    for ts, recs in snapshot:
        for r in recs:
            rows.append({
                "interval_ts": ts,
                "facility_code": r.get("facility_code"),
                "fueltech_id": r.get("fueltech_id"),
                "network_region": r.get("network_region"),
                "power_mw": r.get("power_mw"),
                "emissions_tco2e": r.get("emissions_tco2e"),
            })
    df = pd.DataFrame(rows)
    df["interval_ts"] = pd.to_datetime(df["interval_ts"], utc=True,
                                       errors="coerce")
    df["power_mw"] = pd.to_numeric(df["power_mw"], errors="coerce")
    df["emissions_tco2e"] = pd.to_numeric(df["emissions_tco2e"],
                                          errors="coerce")
    return df.dropna(subset=["interval_ts"])


def market_window(end_ts, hours: int = 6):
    """Return (interval_ts, price, demand) for the trailing window ending at
    `end_ts`. Empty DataFrame if no market data is loaded."""
    if market_df.empty or end_ts is None:
        return pd.DataFrame()
    try:
        end = pd.Timestamp(end_ts)
        start = end - pd.Timedelta(hours=hours)
        sub = market_df.loc[start:end]
        if sub.empty:
            return pd.DataFrame()
        return sub.reset_index()
    except Exception:
        return pd.DataFrame()


In [7]:
# Dash App layout and callbacks

app = Dash(__name__)
app.title = "NEM Live Generation & Emissions"

COLORS = {
    "bg":      "#f4f6fb",   # page background
    "card":    "#ffffff",   # surfaces
    "border":  "#e6e9f0",   # hairlines
    "text":    "#1f2937",   # primary text
    "muted":   "#6b7280",   # secondary text
    "accent":  "#2563eb",   # primary accent (blue)
    "accent2": "#0ea5e9",   # secondary accent
    "good":    "#16a34a",   # renewables / live green
    "warn":    "#f59e0b",   # gas / amber
    "bad":     "#ef4444",   # coal / red
    "violet":  "#9333ea",
}
FONT = "'Segoe UI', system-ui, -apple-system, Roboto, Helvetica, Arial, sans-serif"
CARD = {
    "background": COLORS["card"],
    "border": f"1px solid {COLORS['border']}",
    "borderRadius": "14px",
    "boxShadow": "0 1px 3px rgba(16,24,40,0.06), 0 1px 2px rgba(16,24,40,0.04)",
    "padding": "16px 18px",
}
LABEL = {"fontSize": "12px", "fontWeight": 600, "color": COLORS["muted"],
         "textTransform": "uppercase", "letterSpacing": "0.04em",
         "marginBottom": "6px", "display": "block"}
CARD_TITLE = {"fontSize": "15px", "fontWeight": 700, "color": COLORS["text"],
              "margin": "0 0 10px 0"}

RENEWABLE = {"solar_utility", "solar_rooftop", "solar", "wind", "hydro",
             "bioenergy_biomass", "bioenergy_biogas", "pumps"}

FUEL_GROUP_ORDER = ["Coal", "Gas", "Hydro", "Wind", "Solar",
                    "Bioenergy", "Storage", "Distillate", "Other"]
FUEL_GROUP_COLORS = {
    "Coal": "#374151", "Gas": "#f97316", "Hydro": "#22d3ee",
    "Wind": "#38bdf8", "Solar": "#fbbf24", "Bioenergy": "#84cc16",
    "Storage": "#a78bfa", "Distillate": "#f87171", "Other": "#9ca3af",
}


def _fuel_group(ft):
    if not isinstance(ft, str):
        return "Other"
    if ft.startswith("coal"):
        return "Coal"
    if ft.startswith("gas"):
        return "Gas"
    if ft.startswith("solar"):
        return "Solar"
    if ft.startswith("wind"):
        return "Wind"
    if ft == "hydro":
        return "Hydro"
    if ft.startswith("battery") or ft == "pumps":
        return "Storage"
    if ft.startswith("bioenergy"):
        return "Bioenergy"
    if ft == "distillate":
        return "Distillate"
    return "Other"

FUEL_COLORS = {
    "solar_utility": "#fbbf24", "solar_rooftop": "#fcd34d", "solar": "#fbbf24",
    "wind": "#38bdf8", "hydro": "#22d3ee", "battery": "#a78bfa",
    "battery_charging": "#c4b5fd", "battery_discharging": "#8b5cf6",
    "pumps": "#2dd4bf", "bioenergy_biomass": "#84cc16",
    "bioenergy_biogas": "#a3e635", "gas_ocgt": "#fb923c", "gas_ccgt": "#f97316",
    "gas_steam": "#fdba74", "gas_recip": "#fdba74", "gas_wcmg": "#fed7aa",
    "distillate": "#f87171", "coal_black": "#374151", "coal_brown": "#6b7280",
}


def kpi_card(label, value, accent=COLORS["accent"], sub=None):
    """A KPI tile: small label, big coloured value, optional sub-line."""
    children = [html.Div(label, style=LABEL),
                html.Div(value, style={"fontSize": "22px", "fontWeight": 700,
                                       "color": COLORS["text"]})]
    if sub:
        children.append(html.Div(sub, style={"fontSize": "12px",
                                              "color": COLORS["muted"],
                                              "marginTop": "2px"}))
    return html.Div(style={**CARD, "padding": "12px 16px", "flex": "1",
                           "minWidth": "150px", "borderTop": f"3px solid {accent}"},
                    children=children)


app.layout = html.Div(
    style={"fontFamily": FONT, "background": COLORS["bg"], "minHeight": "100vh",
           "padding": "20px 28px", "color": COLORS["text"]},
    children=[
        # Header
        html.Div(style={"display": "flex", "alignItems": "center",
                        "justifyContent": "space-between", "marginBottom": "18px"},
                 children=[
            html.Div([
                html.H1("NEM Live Generation & Emissions",
                        style={"margin": 0, "fontSize": "26px", "fontWeight": 700}),
                html.Div("Real-time facility stream over MQTT — Group 156",
                         style={"color": COLORS["muted"], "fontSize": "14px",
                                "marginTop": "2px"}),
            ]),
            html.Div(style={"display": "flex", "alignItems": "center", "gap": "8px",
                            "background": "#ecfdf5", "color": COLORS["good"],
                            "border": "1px solid #bbf7d0", "borderRadius": "999px",
                            "padding": "6px 14px", "fontSize": "13px", "fontWeight": 600},
                     children=[html.Span("●", style={"fontSize": "10px"}), "LIVE"]),
        ]),


        html.Div(style={**CARD, "marginBottom": "16px"}, children=[
            html.Div(style={"display": "flex", "flexWrap": "wrap", "gap": "28px",
                            "alignItems": "flex-start"}, children=[
                html.Div(style={"minWidth": "220px"}, children=[
                    html.Label("Metric", style=LABEL),
                    dcc.RadioItems(
                        id="metric",
                        options=[{"label": " Power (MW)", "value": "power_mw"},
                                 {"label": " Emissions (tCO2e)", "value": "emissions_tco2e"}],
                        value="power_mw",
                        labelStyle={"display": "inline-block", "marginRight": "16px",
                                    "fontSize": "14px"},
                        inputStyle={"marginRight": "6px"}),
                ]),
                html.Div(style={"flex": "1", "minWidth": "240px"}, children=[
                    html.Label("Network region", style=LABEL),
                    dcc.Dropdown(id="region", multi=True,
                                 options=[{"label": r, "value": r} for r in REGION_OPTIONS],
                                 placeholder="All regions"),
                ]),
                html.Div(style={"flex": "1", "minWidth": "240px"}, children=[
                    html.Label("Fuel technology", style=LABEL),
                    dcc.Dropdown(id="fuel", multi=True,
                                 options=[{"label": f, "value": f} for f in FUEL_OPTIONS],
                                 placeholder="All fuel types"),
                ]),
            ]),
        ]),


        html.Div(id="status-bar",
                 style={"display": "flex", "gap": "14px", "flexWrap": "wrap",
                        "marginBottom": "16px"}),


        html.Div(style={"display": "flex", "gap": "16px", "flexWrap": "wrap",
                        "marginBottom": "16px"}, children=[
            html.Div(style={**CARD, "padding": "10px", "flex": "2",
                            "minWidth": "420px"}, children=[
                dcc.Graph(id="map", style={"height": "62vh"},
                          config={"displayModeBar": False}),
            ]),
            html.Div(style={"flex": "1", "minWidth": "300px", "display": "flex",
                            "flexDirection": "column", "gap": "16px"}, children=[
                html.Div(style={**CARD}, children=[
                    html.H3("Generation mix by fuel", style=CARD_TITLE),
                    dcc.Graph(id="fuel-chart", style={"height": "27vh"},
                              config={"displayModeBar": False}),
                ]),
                html.Div(style={**CARD}, children=[
                    html.H3("Output by region", style=CARD_TITLE),
                    dcc.Graph(id="region-chart", style={"height": "27vh"},
                              config={"displayModeBar": False}),
                ]),
            ]),
        ]),

        html.Div(style={"display": "flex", "gap": "16px", "flexWrap": "wrap"},
                 children=[
            html.Div(style={**CARD, "flex": "1", "minWidth": "320px"}, children=[
                html.H3("Top facilities", style=CARD_TITLE),
                html.Div(id="leaderboard"),
            ]),
            html.Div(id="detail", style={**CARD, "flex": "1", "minWidth": "320px",
                                         "fontSize": "14px"}),
        ]),

        html.Div(style={"display": "flex", "gap": "16px", "flexWrap": "wrap",
                        "marginTop": "16px"}, children=[
            html.Div(style={**CARD, "flex": "2", "minWidth": "420px"}, children=[
                html.H3("Generation mix — rolling window",
                        style=CARD_TITLE),
                dcc.Graph(id="fuel-mix-trend", style={"height": "28vh"},
                          config={"displayModeBar": False}),
            ]),
            html.Div(style={**CARD, "flex": "1", "minWidth": "320px"}, children=[
                html.H3("Emissions intensity (tCO₂/MWh)",
                        style=CARD_TITLE),
                dcc.Graph(id="intensity-trend", style={"height": "28vh"},
                          config={"displayModeBar": False}),
            ]),
        ]),

        html.Div(style={**CARD, "marginTop": "16px"}, children=[
            html.H3("NEM price & demand — last 6 hours",
                    style=CARD_TITLE),
            dcc.Graph(id="price-demand-trend", style={"height": "26vh"},
                      config={"displayModeBar": False}),
        ]),

        dcc.Interval(id="tick", interval=1500, n_intervals=0),
    ])


def _style_fig(fig, h_margin=False):
    fig.update_layout(margin=dict(l=8, r=8, t=8, b=8),
                      paper_bgcolor=COLORS["card"], plot_bgcolor=COLORS["card"],
                      font=dict(family=FONT, color=COLORS["text"], size=12),
                      showlegend=False)
    return fig


def _empty_map():
    fig = px.scatter_map(lat=[], lon=[], zoom=3.5,
                         center={"lat": -33.5, "lon": 146},
                         map_style="open-street-map")
    fig.update_layout(margin=dict(l=0, r=0, t=0, b=0),
                      paper_bgcolor=COLORS["card"], font=dict(family=FONT))
    return fig


def _empty_small(text="Waiting for data…"):
    fig = go.Figure()
    fig.add_annotation(text=text, showarrow=False,
                       font=dict(color=COLORS["muted"], size=13))
    fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)
    return _style_fig(fig)


@app.callback(
    Output("map", "figure"),
    Output("status-bar", "children"),
    Output("fuel-chart", "figure"),
    Output("region-chart", "figure"),
    Output("leaderboard", "children"),
    Output("fuel-mix-trend", "figure"),
    Output("intensity-trend", "figure"),
    Output("price-demand-trend", "figure"),
    Input("tick", "n_intervals"),
    Input("metric", "value"),
    Input("region", "value"),
    Input("fuel", "value"),
)
def refresh(_n, metric, regions, fuels):
    df, n_recv, ev_ts = snapshot_df()
    price, demand = market_now(ev_ts)
    metric_label = "Power (MW)" if metric == "power_mw" else "Emissions (tCO2e)"

    def kpis(shown, gen=0.0, emit=0.0, clean=None):
        clean_txt = f"{clean:.0f}%" if clean is not None else "—"
        return [
            kpi_card("Total generation", f"{gen:,.0f} MW", COLORS["accent"],
                     f"{shown:,} facilities shown"),
            kpi_card("Total emissions", f"{emit:,.0f} tCO2e", COLORS["bad"],
                     "per 5-min interval"),
            kpi_card("Clean energy share", clean_txt, COLORS["good"],
                     "solar · wind · hydro · bio"),
            kpi_card("Messages received", f"{n_recv:,}", COLORS["accent2"],
                     (f"@ {ev_ts.replace('T', ' ')[:16]}" if ev_ts else "waiting")),
            kpi_card("NEM price",
                     (f"${price:,.0f}/MWh" if price is not None else "—"),
                     COLORS["warn"],
                     (f"demand {demand:,.0f} MW" if demand is not None else None)),
        ]

    if df.empty:
        return (_empty_map(), kpis(0), _empty_small(), _empty_small(),
                html.Span("Waiting for messages…", style={"color": COLORS["muted"]}),
                _empty_small("Waiting for stream…"),
                _empty_small("Waiting for stream…"),
                _empty_small("Waiting for stream…"))

    if regions:
        df = df[df["network_region"].isin(regions)]
    if fuels:
        df = df[df["fueltech_id"].isin(fuels)]
    if df.empty:
        return (_empty_map(), kpis(0), _empty_small("No match"),
                _empty_small("No match"),
                html.Span("No facilities match the current filters.",
                          style={"color": COLORS["muted"]}),
                _empty_small("No match"),
                _empty_small("No match"),
                _empty_small("No match"))

    pos = df["power_mw"].clip(lower=0).fillna(0)
    total_gen = float(pos.sum())
    total_emit = float(df["emissions_tco2e"].fillna(0).sum())
    clean_mask = df["fueltech_id"].isin(RENEWABLE)
    clean_gen = float(pos[clean_mask].sum())
    clean_share = (100.0 * clean_gen / total_gen) if total_gen > 0 else 0.0

    df = df.copy()
    df["_size"] = df[metric].abs().fillna(0.0) + 1.0
    if "capacity_registered_mw" not in df.columns:
        df["capacity_registered_mw"] = np.nan
    df["capacity_registered_mw"] = pd.to_numeric(
        df["capacity_registered_mw"], errors="coerce")
    fig_map = px.scatter_map(
        df, lat="lat", lon="lon", color=metric, size="_size", size_max=26,
        color_continuous_scale=COLOR_SCALES.get(metric, "Viridis"),
        # facility_code MUST stay at customdata[0] - the marker-click callback
        # (show_detail) reads pt["customdata"][0] to identify the station.
        custom_data=["facility_code", "facility_name", "fueltech_id",
                     "network_region", "power_mw", "emissions_tco2e",
                     "capacity_registered_mw"],
        zoom=3.8, center={"lat": -33.5, "lon": 146},
        map_style="open-street-map", labels={metric: metric_label})
    # Clean, labelled tooltip shown on mouse-over of any station marker.
    fig_map.update_traces(hovertemplate=(
        "<b>%{customdata[1]}</b><br>"
        "<span style='color:#9ca3af'>%{customdata[0]} · %{customdata[2]}"
        " · %{customdata[3]}</span><br>"
        "Power: <b>%{customdata[4]:,.1f} MW</b><br>"
        "Emissions: <b>%{customdata[5]:,.2f} tCO₂e</b><br>"
        "Capacity: %{customdata[6]:,.0f} MW"
        "<extra></extra>"))
    fig_map.update_layout(
        margin=dict(l=0, r=0, t=0, b=0), paper_bgcolor=COLORS["card"],
        font=dict(family=FONT, color=COLORS["text"]),
        coloraxis_colorbar=dict(title=metric_label, thickness=12, len=0.7,
                                xpad=6, tickfont=dict(size=11)),
        hoverlabel=dict(bgcolor="white", font_size=12, font_family=FONT),
        uirevision="constant")

    by_fuel = (df.assign(_v=df[metric].clip(lower=0).fillna(0))
                 .groupby("fueltech_id", as_index=False)["_v"].sum())
    by_fuel = by_fuel[by_fuel["_v"] > 0].sort_values("_v", ascending=False)
    if by_fuel.empty:
        fig_fuel = _empty_small("No positive output")
    else:
        fig_fuel = px.pie(by_fuel, names="fueltech_id", values="_v", hole=0.55,
                          color="fueltech_id", color_discrete_map=FUEL_COLORS)
        fig_fuel.update_traces(textposition="inside", textinfo="percent",
                               sort=False)
        fig_fuel = _style_fig(fig_fuel)
        fig_fuel.update_layout(showlegend=True,
                               legend=dict(orientation="v", x=1.02, y=0.5,
                                           font=dict(size=10)))

    by_region = (df.assign(_v=df[metric].clip(lower=0).fillna(0))
                   .groupby("network_region", as_index=False)["_v"].sum()
                   .sort_values("_v"))
    if by_region.empty:
        fig_region = _empty_small()
    else:
        fig_region = px.bar(by_region, x="_v", y="network_region",
                            orientation="h", text="_v",
                            color_discrete_sequence=[COLORS["accent"]])
        fig_region.update_traces(texttemplate="%{text:,.0f}",
                                 textposition="outside", cliponaxis=False)
        fig_region = _style_fig(fig_region)
        fig_region.update_layout(
            xaxis_title=metric_label, yaxis_title=None,
            xaxis=dict(gridcolor=COLORS["border"], zeroline=False),
            yaxis=dict(showgrid=False))

    top = df.sort_values(metric, ascending=False).head(8)
    rows = []
    for i, (_, r) in enumerate(top.iterrows(), start=1):
        dot = FUEL_COLORS.get(r["fueltech_id"], COLORS["muted"])
        val = r["power_mw"] if metric == "power_mw" else r["emissions_tco2e"]
        unit = "MW" if metric == "power_mw" else "tCO2e"
        rows.append(html.Tr([
            html.Td(f"{i}", style={"color": COLORS["muted"], "width": "26px"}),
            html.Td([html.Span("●", style={"color": dot, "marginRight": "8px"}),
                     r["facility_name"]],
                    style={"fontWeight": 600}),
            html.Td(r["network_region"], style={"color": COLORS["muted"]}),
            html.Td(f"{val:,.1f} {unit}",
                    style={"textAlign": "right", "fontWeight": 700}),
        ], style={"borderBottom": f"1px solid {COLORS['border']}"}))
    leaderboard = html.Table(style={"width": "100%", "borderCollapse": "collapse",
                                    "fontSize": "13px"},
                             children=[html.Tbody(rows)])

    # --- Time-series charts (rolling history) -------------------------------
    hist = history_df()
    if not hist.empty:
        if regions:
            hist = hist[hist["network_region"].isin(regions)]
        if fuels:
            hist = hist[hist["fueltech_id"].isin(fuels)]

    if hist.empty:
        fig_mix_trend = _empty_small("Stream warming up…")
        fig_intensity = _empty_small("Stream warming up…")
    else:
        hist = hist.copy()
        hist["fuel_group"] = hist["fueltech_id"].map(_fuel_group)
        # Stacked area: total MW by fuel group at each event time.
        mix_ts = (hist.assign(p=hist["power_mw"].clip(lower=0).fillna(0))
                       .groupby(["interval_ts", "fuel_group"], as_index=False)["p"].sum())
        wide = (mix_ts.pivot(index="interval_ts", columns="fuel_group",
                              values="p").fillna(0.0).sort_index())
        cols = [c for c in FUEL_GROUP_ORDER if c in wide.columns]
        wide = wide[cols]
        fig_mix_trend = go.Figure()
        for col in cols:
            fig_mix_trend.add_trace(go.Scatter(
                x=wide.index, y=wide[col], mode="lines",
                name=col, stackgroup="one",
                line=dict(width=0.4, color=FUEL_GROUP_COLORS.get(col, "#9ca3af")),
                fillcolor=FUEL_GROUP_COLORS.get(col, "#9ca3af"),
                hovertemplate=f"{col}: %{{y:,.0f}} MW<extra></extra>",
            ))
        fig_mix_trend = _style_fig(fig_mix_trend)
        fig_mix_trend.update_layout(
            showlegend=True,
            legend=dict(orientation="h", yanchor="bottom", y=1.02,
                        xanchor="left", x=0, font=dict(size=10)),
            xaxis=dict(gridcolor=COLORS["border"], title=None),
            yaxis=dict(title="Power (MW)", gridcolor=COLORS["border"]),
            hovermode="x unified",
        )

        # Emissions intensity (tCO2/MWh) at each event time = total emissions
        # divided by total energy (MW / 12 for the 5-min slot).
        per_ts = (hist.assign(p=hist["power_mw"].clip(lower=0).fillna(0),
                                e=hist["emissions_tco2e"].fillna(0))
                       .groupby("interval_ts", as_index=False)
                       .agg(power_mw=("p", "sum"),
                            emissions_tco2e=("e", "sum")))
        per_ts["energy_mwh"] = per_ts["power_mw"] / 12.0
        per_ts["intensity"] = (per_ts["emissions_tco2e"]
                                / per_ts["energy_mwh"].replace(0, np.nan))
        per_ts = per_ts.dropna(subset=["intensity"]).sort_values("interval_ts")
        if per_ts.empty:
            fig_intensity = _empty_small("Waiting for thermal output…")
        else:
            fig_intensity = go.Figure(go.Scatter(
                x=per_ts["interval_ts"], y=per_ts["intensity"], mode="lines",
                line=dict(color=COLORS["bad"], width=2),
                fill="tozeroy", fillcolor="rgba(239,68,68,0.10)",
                hovertemplate="%{y:.3f} tCO₂/MWh<extra></extra>",
            ))
            fig_intensity = _style_fig(fig_intensity)
            fig_intensity.update_layout(
                xaxis=dict(gridcolor=COLORS["border"], title=None),
                yaxis=dict(title="tCO₂/MWh", gridcolor=COLORS["border"]),
                hovermode="x unified",
            )

    # Price + demand twin-axis chart from the optional market series.
    mkt = market_window(ev_ts, hours=6) if ev_ts else pd.DataFrame()
    if mkt.empty:
        fig_pd = _empty_small("No market data" if market_df.empty
                               else "Waiting for stream…")
    else:
        fig_pd = go.Figure()
        if "demand" in mkt.columns:
            fig_pd.add_trace(go.Scatter(
                x=mkt["interval_ts"], y=mkt["demand"], name="Demand (MW)",
                mode="lines", line=dict(color=COLORS["accent"], width=2),
                yaxis="y1",
                hovertemplate="Demand: %{y:,.0f} MW<extra></extra>"))
        if "price" in mkt.columns:
            fig_pd.add_trace(go.Scatter(
                x=mkt["interval_ts"], y=mkt["price"], name="Price ($/MWh)",
                mode="lines", line=dict(color=COLORS["warn"], width=2, dash="dot"),
                yaxis="y2",
                hovertemplate="Price: $%{y:,.0f}/MWh<extra></extra>"))
        fig_pd = _style_fig(fig_pd)
        fig_pd.update_layout(
            showlegend=True,
            legend=dict(orientation="h", yanchor="bottom", y=1.02,
                        xanchor="left", x=0, font=dict(size=10)),
            xaxis=dict(gridcolor=COLORS["border"], title=None),
            yaxis=dict(title="Demand (MW)", gridcolor=COLORS["border"]),
            yaxis2=dict(title="Price ($/MWh)", overlaying="y", side="right",
                        showgrid=False),
            hovermode="x unified",
        )

    return (fig_map, kpis(len(df), total_gen, total_emit, clean_share),
            fig_fuel, fig_region, leaderboard,
            fig_mix_trend, fig_intensity, fig_pd)


@app.callback(
    Output("detail", "children"),
    Input("map", "clickData"),
)
def show_detail(click):
    head = html.H3("Station detail", style=CARD_TITLE)
    if not click or "points" not in click or not click["points"]:
        return [head, html.Span("Click any marker on the map to inspect a station.",
                                style={"color": COLORS["muted"]})]
    pt = click["points"][0]
    fc = pt["customdata"][0] if pt.get("customdata") else None
    with STATE_LOCK:
        rec = latest_by_facility.get(fc)
    if not rec:
        return [head, html.Span("No data for the selected station yet.",
                                style={"color": COLORS["muted"]})]

    def chip(text, bg, fg):
        return html.Span(text, style={
            "background": bg, "color": fg, "borderRadius": "999px",
            "padding": "3px 10px", "fontSize": "12px", "fontWeight": 600,
            "marginRight": "8px"})

    def stat(label, value):
        return html.Div(style={"marginRight": "28px"}, children=[
            html.Div(label, style=LABEL),
            html.Div(value, style={"fontSize": "18px", "fontWeight": 700}),
        ])

    return [head,
        html.Div(style={"display": "flex", "alignItems": "baseline", "gap": "10px",
                        "marginBottom": "12px", "flexWrap": "wrap"}, children=[
            html.Span(rec.get("facility_name", fc),
                      style={"fontSize": "18px", "fontWeight": 700}),
            chip(rec.get("facility_code", ""), "#eff6ff", COLORS["accent"]),
            chip(str(rec.get("fueltech_id", "")), "#f0fdf4", COLORS["good"]),
            chip(str(rec.get("network_region", "")), "#faf5ff", COLORS["violet"]),
        ]),
        html.Div(style={"display": "flex", "flexWrap": "wrap"}, children=[
            stat("Power", f"{rec.get('power_mw')} MW"),
            stat("Emissions", f"{rec.get('emissions_tco2e')} tCO2e"),
            stat("Capacity", f"{rec.get('capacity_registered_mw')} MW"),
        ]),
        html.Div(f"Event time: {rec.get('event_time')}",
                 style={"color": COLORS["muted"], "fontSize": "12px",
                        "marginTop": "10px"}),
    ]


In [8]:

print(f"Starting Dash on http://127.0.0.1:{DASH_PORT}  (Ctrl-C / interrupt to stop)")
app.run(debug=False, port=DASH_PORT, jupyter_mode="external")

Starting Dash on http://127.0.0.1:8050  (Ctrl-C / interrupt to stop)
Dash app running on http://127.0.0.1:8050/


[subscriber] connected rc=Success, subscribing to comp5339/group156/electricity


## Notes

- **Concurrency:** the MQTT subscriber runs on a background thread; the Dash
  `dcc.Interval` polls the shared state every 1.5 s and redraws the map. This
  keeps the UI responsive while messages stream in.
- **Integration with Assignment 1:** with `INTEGRATE_TO_DUCKDB = True`, every
  received message is matched to its `FACILITY` row by `facility_code` and inserted
  into `FACILITY_POWER_EMISSIONS` in the shared `electricity_a2.duckdb` — the same
  normalised schema designed in Assignment 1.
- **Stopping:** interrupt the kernel to stop the server; the subscriber loop and
  DuckDB connection are cleaned up automatically when the kernel exits. To free the
  DuckDB write lock for the publisher notebook, restart this kernel.
- **Market price/demand** come from the optional Task 1 market series
  (`market_price_demand_5min.csv`), looked up at the current stream event time.